# Pale — PyTorch transfer learning cross-run benchmark

Tests Pale's cross-run deduplication for the transfer learning scenario:
a single pretrained ResNet-18 base is fine-tuned in N independent experiments.
All runs share the same frozen base layer weights — Pale should store those
chunks exactly once, regardless of how many runs reference them.

**Setup:**
- Base: `ResNet18_Weights.DEFAULT` (pretrained ImageNet weights, ~45 MB)
- Fine-tuning: 4 independent runs with different seeds and learning rates
- Each run: 10 epochs, FC layer only (backbone fully frozen throughout)

**Key questions:**
1. What fraction of chunks are shared across fine-tuning runs?
2. How much does the second/third/fourth run cost in new bytes written?
3. What is the no-op rate within each run?
4. How does Pale storage compare to DVC across all 4 runs combined?

In [ ]:
!pip install -q git+https://github.com/Olamyy/pale.git@hash-cache-no-op-path zstandard torch torchvision

In [ ]:
import sys
import time
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(Path(".").resolve()))
from utils import (
    CHUNK_SIZE,
    extract_pytorch,
    measure_noop,
    measure_chunk_dedup,
    measure_crossrun,
    dvc_bytes,
    pale_bytes,
    print_noop,
    print_chunk,
    print_crossrun,
    print_dvc_comparison,
    _fmt_bytes,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("Imports OK")

## Configuration

In [ ]:
N_EPOCHS = 10
NUM_CLASSES = 10
CHECKPOINT_DIR = Path("/tmp/pale_pytorch_transfer")

# Four fine-tuning runs: vary seed and learning rate to simulate real experiments
RUNS = [
    {"run_id": "run_a", "seed": 42,  "lr": 1e-3},
    {"run_id": "run_b", "seed": 99,  "lr": 1e-3},
    {"run_id": "run_c", "seed": 7,   "lr": 5e-4},
    {"run_id": "run_d", "seed": 123, "lr": 2e-3},
]

print(f"Epochs per run: {N_EPOCHS}")
print(f"Runs: {len(RUNS)}")
print(f"Total checkpoints: {N_EPOCHS * len(RUNS)}")

## Model setup

Uses `ResNet18_Weights.DEFAULT` (pretrained ImageNet weights). The FC layer is
replaced to match `NUM_CLASSES`. The backbone is fully frozen from epoch 1 —
only the FC layer trains. This is the canonical transfer learning setup:
pretrained base + task-specific head.

All four runs start from the same pretrained weights. The only things that differ
across runs are the FC layer initialization (via seed) and the learning rate.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

def _make_pretrained_resnet18(num_classes: int = NUM_CLASSES) -> nn.Module:
    """Load pretrained ResNet-18, replace FC, freeze backbone."""
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    # Freeze everything except the FC layer
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad_(False)
    return model.to(device)


def _count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Inspect the model before training
sample = _make_pretrained_resnet18()
total, trainable = _count_params(sample)
state = sample.state_dict()
tensor_bytes = sum(v.numel() * v.element_size() for v in state.values())
print(f"ResNet-18 (pretrained, {NUM_CLASSES} classes)")
print(f"  Parameters : {total:,} total, {trainable:,} trainable ({100*trainable/total:.2f}%)")
print(f"  State dict : {len(state)} tensors, {_fmt_bytes(tensor_bytes)} uncompressed")
print(f"  Frozen     : {total - trainable:,} parameters (entire backbone)")
del sample

## Training

Each run fine-tunes a fresh copy of the pretrained ResNet-18 with its own seed
and learning rate. The backbone is frozen throughout — only the FC layer updates.

Synthetic 32×32 images are used (no ImageNet download required).

In [ ]:
def _make_dataset(seed: int):
    rng = np.random.default_rng(seed)
    X = torch.from_numpy(rng.standard_normal((512, 3, 32, 32)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, NUM_CLASSES, 512).astype(np.int64)).to(device)
    return torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True, num_workers=0,
    )


def fine_tune(
    seed: int,
    lr: float,
    n_epochs: int = N_EPOCHS,
) -> list[dict]:
    """Fine-tune pretrained ResNet-18 (frozen backbone, FC only).
    Returns one state dict per epoch.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = _make_pretrained_resnet18()
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss()
    loader = _make_dataset(seed)

    state_dicts = []
    model.train()
    for epoch in range(1, n_epochs + 1):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})

    return state_dicts


print("Training helpers OK")

In [ ]:
all_state_dicts = {}

for run in RUNS:
    t0 = time.time()
    print(f"Training {run['run_id']} (seed={run['seed']}, lr={run['lr']})...", end=" ", flush=True)
    all_state_dicts[run["run_id"]] = fine_tune(seed=run["seed"], lr=run["lr"])
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints trained: {sum(len(v) for v in all_state_dicts.values())}")

## Extract tensors

In [ ]:
all_seqs = {
    run_id: [extract_pytorch(sd) for sd in sds]
    for run_id, sds in all_state_dicts.items()
}

sample_seq = all_seqs["run_a"]
n_tensors = len(sample_seq[0])
total_bytes = sum(v.nbytes for v in sample_seq[-1].values())
print(f"Tensors per checkpoint : {n_tensors}")
print(f"Tensor bytes (run_a epoch 10) : {_fmt_bytes(total_bytes)}")

## No-op fast path — within each run

With the backbone fully frozen from epoch 1, we expect:
- Conv weights, BN weight/bias: 100% no-op (frozen parameters, never updated)
- BN running stats: 0% no-op (updated every forward pass in train mode)
- FC weight/bias: 0% no-op (the only trainable layer)

All four runs should show the same pattern since the frozen set is identical.

In [ ]:
for run_id, seqs in all_seqs.items():
    noop_stats = measure_noop(seqs)
    overall = noop_stats["__summary__"]["identical_pct"]
    print(f"{run_id}: {overall:.1f}% overall no-op")

print()
# Detailed breakdown for run_a
print_noop("run_a (detailed)", measure_noop(all_seqs["run_a"]), max_tensors=20)

## Cross-run chunk sharing

This is the core experiment. All four runs start from the same pretrained weights.
The frozen backbone layers (conv weights, BN weight/bias) are identical across
all runs — they never update. Only the FC layer and BN running stats differ.

Expected:
- High sharing (~80–90%) between any two runs: the backbone chunks are shared
- Sharing does not degrade with more runs: each additional run writes only FC + BN stats

We measure pairwise (run_a vs each other run) and then cumulative
(how many new chunks does each successive run add to the store).

In [ ]:
print("Pairwise cross-run chunk sharing vs run_a:\n")
run_ids = list(all_seqs.keys())
for run_id in run_ids[1:]:
    stats = measure_crossrun(all_seqs["run_a"], all_seqs[run_id], CHUNK_SIZE)
    print_crossrun(f"run_a vs {run_id}", stats)

In [ ]:
# Cumulative: how many new unique chunks does each run add to the store?
# Simulates a shared PaleStore where all runs write into the same CAS.

from pale.hashing import hash_chunk as _hash
from pale.chunking import chunk_bytes as _chunk
from pale.serialization import tensor_to_bytes as _tensor_to_bytes

def _unique_hashes(seqs) -> set:
    s = set()
    for tensors in seqs:
        for arr in tensors.values():
            raw, _, _ = _tensor_to_bytes(arr)
            for c in _chunk(raw, CHUNK_SIZE):
                s.add(_hash(c))
    return s


print(f"{'Run':<8} {'New chunks':>12} {'Cumulative':>12} {'New bytes (est)':>16} {'Marginal cost':>14}")
print("=" * 66)

cumulative: set = set()
for run in RUNS:
    run_id = run["run_id"]
    hashes = _unique_hashes(all_seqs[run_id])
    new = hashes - cumulative
    cumulative |= hashes
    new_bytes_est = len(new) * CHUNK_SIZE
    marginal_pct = len(new) / len(hashes) * 100 if hashes else 0
    print(f"  {run_id:<6} {len(new):>12,} {len(cumulative):>12,} {_fmt_bytes(new_bytes_est):>16} {marginal_pct:>13.1f}%")

## Pale shared store — actual bytes written

Saves all four runs into a single `PaleStore` instance (shared CAS root).
Measures how many bytes are written to disk after each run is added.

This is the real storage cost, not an estimate: actual `.chunk` file sizes
on disk after zstd compression.

In [ ]:
from pale.store import PaleStore
from pale.adapters.pytorch import PyTorchAdapter


def _sd_to_model(sd):
    m = _make_pretrained_resnet18()
    m.load_state_dict({k: v.clone() for k, v in sd.items()})
    return m


print(f"{'Run':<8} {'Checkpoints':>12} {'Store size after':>17} {'New bytes':>12} {'Marginal cost':>14}")
print("=" * 68)

with tempfile.TemporaryDirectory() as tmp:
    store_root = Path(tmp)
    prev_size = 0

    for run in RUNS:
        run_id = run["run_id"]
        models = [_sd_to_model(sd) for sd in all_state_dicts[run_id]]

        with PaleStore(root=store_root, run_id=run_id, adapter=PyTorchAdapter()) as store:
            for step, model in enumerate(models, 1):
                store.save(model, step=step)

        current_size = pale_bytes(store_root)
        new_bytes = current_size - prev_size
        marginal_pct = new_bytes / current_size * 100 if current_size else 0
        print(f"  {run_id:<6} {N_EPOCHS:>12} {_fmt_bytes(current_size):>17} {_fmt_bytes(new_bytes):>12} {marginal_pct:>13.1f}%")
        prev_size = current_size

## DVC vs Pale — all 4 runs combined

DVC stores one compressed file per checkpoint per run — 40 files total.
It has no cross-run deduplication; each run's files are independent.

Pale stores all 40 checkpoints in a shared CAS. Identical chunks across
runs (the frozen backbone) are stored once regardless of how many runs
reference them.

This section measures the total storage cost of all 4 runs combined.

In [ ]:
import torch

# Write one .pt file per checkpoint per run
checkpoint_files_all = []
for run in RUNS:
    run_id = run["run_id"]
    run_dir = CHECKPOINT_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    for epoch, sd in enumerate(all_state_dicts[run_id], 1):
        path = run_dir / f"epoch_{epoch:06d}.pt"
        torch.save(sd, path)
        checkpoint_files_all.append(path)

total_raw = sum(p.stat().st_size for p in checkpoint_files_all)
print(f"{len(checkpoint_files_all)} checkpoints written")
print(f"Raw total (all runs): {_fmt_bytes(total_raw)}")

In [ ]:
# DVC: file-level dedup across all 40 checkpoints
dvc_total = dvc_bytes(checkpoint_files_all)
print(f"DVC total (all 4 runs, file-level dedup): {_fmt_bytes(dvc_total)}")
print(f"  Per run avg: {_fmt_bytes(dvc_total / len(RUNS))}")
print()

# Pale: tensor-level dedup across all 40 checkpoints in a shared store
with tempfile.TemporaryDirectory() as tmp:
    store_root = Path(tmp)
    for run in RUNS:
        run_id = run["run_id"]
        models = [_sd_to_model(sd) for sd in all_state_dicts[run_id]]
        with PaleStore(root=store_root, run_id=run_id, adapter=PyTorchAdapter()) as store:
            for step, model in enumerate(models, 1):
                store.save(model, step=step)
    pale_total = pale_bytes(store_root)

savings = (dvc_total - pale_total) / dvc_total * 100 if dvc_total else 0
print(f"Pale total (all 4 runs, tensor-level dedup): {_fmt_bytes(pale_total)}")
print()
print(f"{'':>30} {'DVC':>12} {'Pale':>12} {'Savings':>10}")
print("=" * 68)
print(f"  {'4 runs × 10 epochs':<28} {_fmt_bytes(dvc_total):>12} {_fmt_bytes(pale_total):>12} {savings:>9.1f}%")

## Which layers are shared and which are unique per run?

Breaks down chunk sharing by layer group across all four runs.
Frozen backbone layers should show 100% sharing; FC and BN running stats should show 0%.

In [ ]:
# For each tensor name, collect all unique chunk hashes across all runs and all steps
from collections import defaultdict

tensor_hashes_per_run: dict[str, dict[str, set]] = defaultdict(lambda: defaultdict(set))

for run_id, seqs in all_seqs.items():
    for tensors in seqs:
        for name, arr in tensors.items():
            raw, _, _ = _tensor_to_bytes(arr)
            for c in _chunk(raw, CHUNK_SIZE):
                tensor_hashes_per_run[name][run_id].add(_hash(c))

# For each tensor: are its chunks identical across all 4 runs?
tensor_names = sorted(tensor_hashes_per_run.keys())
run_ids = [r["run_id"] for r in RUNS]

print(f"  {'Tensor':<45} {'Shared across all runs':>22} {'Per-run unique':>15}")
print(f"  {'-'*45} {'-'*22} {'-'*15}")

for name in tensor_names:
    per_run = tensor_hashes_per_run[name]
    all_hashes = set.union(*per_run.values())
    shared = set.intersection(*per_run.values())
    shared_pct = len(shared) / len(all_hashes) * 100 if all_hashes else 0
    unique_per_run = len(all_hashes) - len(shared)
    print(f"  {name:<45} {shared_pct:>20.1f}% {unique_per_run:>15,}")

## Summary

| Question | Expected | Result |
|---|---|---|
| No-op rate within each run | High (frozen backbone) | TBD |
| Chunk sharing run_a vs run_b | High (same frozen backbone) | TBD |
| Marginal cost of run 2, 3, 4 | Near-zero (only FC + BN stats new) | TBD |
| Pale vs DVC across all 4 runs | Much better (backbone stored once) | TBD |
| Shared layers | All frozen layers (conv, BN params) | TBD |
| Unique per-run layers | FC, BN running stats | TBD |